# Лабораторная 04. Spark UI как инструмент диагностики

Цель: научиться читать Jobs, Stages, SQL и Executors tabs.

In [ ]:
from pathlib import Path
import shutil
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder.appName('lab-04-spark-ui').master('local[*]')
    .config('spark.driver.memory', '2g')
    .config('spark.sql.shuffle.partitions', '8')
    .config('spark.sql.adaptive.enabled', 'false')
    .getOrCreate())
spark.sparkContext.setLogLevel('WARN')
base = Path('spark_core_data').absolute()
base_uri = base.as_uri()
orders = spark.read.parquet(f'{base_uri}/orders')
customers = spark.read.parquet(f'{base_uri}/customers')
print('Spark UI:', spark.sparkContext.uiWebUrl)

## Операция 1: count
Запустите action и заполните Jobs/Stages.

In [ ]:
orders.count()

## Операция 2: groupBy
Смотрите Shuffle Read/Write и SQL plan.

In [ ]:
orders.groupBy('status').agg(F.count('*').alias('cnt'), F.sum('order_amount').alias('amount')).show()

## Операция 3: join
Отключим broadcast, чтобы план был показательнее.

In [ ]:
spark.conf.set('spark.sql.autoBroadcastJoinThreshold', '-1')
orders.join(customers, 'customer_id').groupBy('region').count().show()

## Операция 4: repartition + write
Запись тоже action. Количество output files связано с количеством partitions.

In [ ]:
out = base / 'ui_lab_output'
if out.exists():
    shutil.rmtree(out)
orders.repartition(6).write.mode('overwrite').parquet(out.as_uri())
out.as_uri()

Заполните для каждой операции:

| Операция | Job ID | Stages | Duration | Самый долгий stage | Tasks | Shuffle Read/Write | Spill | Оператор в SQL plan |
|---|---|---:|---:|---|---:|---|---|---|
| count | ? | ? | ? | ? | ? | ? | ? | ? |
| groupBy | ? | ? | ? | ? | ? | ? | ? | ? |
| join | ? | ? | ? | ? | ? | ? | ? | ? |
| repartition/write | ? | ? | ? | ? | ? | ? | ? | ? |

Итоговый мини-отчёт 5-7 предложений:

```text
Самым дорогим оказался stage ...
Он дорогой, потому что ...
Я понял это по ...
В плане был оператор ...
```